# Lab 3 — Camera geometry: from 3D points to pixels (and back)

**Deep Learning for Computer Vision — Day 1** · ECAM, 5MAU · 2026–2027

**Goals.**
1. Project 3D points with the pinhole model in homogeneous coordinates.
2. Build the projection matrix $P = K\,[R \mid t]$ from the intrinsics and the camera pose; play with them.
3. Model the lens distortion.
4. Go back from a pixel to a point on the table plane (back-projection).
5. Estimate a **homography** from 4 points (DLT) and use it to rectify a photo.

**Duration:** ~45 min. Sections 1–3 and 5 are the core; 4 and 6 can be finished at home.

> *Sections 1 and 5 are adapted from* Q. Lurkin, *Image Processing – Lab 1: Camera Geometry*, ECAM, https://quentin.lurkin.xyz/courses/imageprocessing/lab1/ (cube, homography from 4 points, `cover.jpg`). The bonus (section 6) is the final exercise of Q. Lurkin, *Lab 2*, with its image `document.jpg`. Other references: R. Hartley & A. Zisserman, *Multiple View Geometry in Computer Vision*, 2nd ed., 2004, ch. 6; R. Szeliski, *Computer Vision: Algorithms and Applications*, 2nd ed., ch. 2.

In [ ]:
# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "cv2": "opencv-contrib-python",
            "ipywidgets": "ipywidgets", "ipympl": "ipympl", "serial": "pyserial"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if "opencv-contrib-python" not in missing:
    import cv2
    if not hasattr(cv2, "aruco"):  # opencv-python (without contrib) shadows opencv-contrib-python
        missing.append("opencv-contrib-python (run `%pip uninstall -y opencv-python opencv-contrib-python` "
                       "then `%pip install \"opencv-contrib-python<5\"`)")
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

DATA = Path("data") if Path("data").exists() else Path("../data")
np.set_printoptions(precision=3, suppress=True)

## 1. The pinhole model in homogeneous coordinates

A 3D point $(X, Y, Z)$ expressed in the **camera frame** (z along the optical axis) is projected on the image plane at distance $f$:

$$ x = f\,\frac{X}{Z}, \qquad y = f\,\frac{Y}{Z}. $$

The division by $Z$ is not linear, but it becomes a matrix product in **homogeneous coordinates**: a 2D point $(x, y)$ is represented by any vector $(sx, sy, s)$ with $s \neq 0$.

$$ \begin{pmatrix} sx \\ sy \\ s \end{pmatrix} = \begin{pmatrix} f & 0 & 0 & 0 \\ 0 & f & 0 & 0 \\ 0 & 0 & 1 & 0 \end{pmatrix} \begin{pmatrix} X \\ Y \\ Z \\ 1 \end{pmatrix} $$

These are the same homogeneous coordinates as the 4×4 transformation matrices of your robotics course.

### Exercise 1.1 — Project a cube
The vertices of a cube of side 2 centred at the origin are given below (some are repeated to draw the edges with one `plot`). Move the cube 6 units in front of the camera, project it with $f = 1$ and draw it.

In [ ]:
x = [1, -1, -1, 1, 1, 1, -1, -1, 1, 1, -1, -1, -1, -1, 1, 1]
y = [1, 1, -1, -1, 1, 1, 1, -1, -1, 1, 1, 1, -1, -1, -1, -1]
z = [1, 1, 1, 1, 1, -1, -1, -1, -1, -1, -1, 1, 1, -1, -1, 1]
Xcube = np.array([x, y, z, [1] * 16], dtype=float)  # 4 x 16, homogeneous


def to_euclidean(ph):
    """Homogeneous (n+1) x N -> Euclidean n x N."""
    return ph[:-1] / ph[-1]


# TODO
...

plt.figure(figsize=(4, 4))
plt.plot(xy[0], xy[1], "-o")
plt.gca().set_aspect("equal")
plt.title("cube seen from the front");

## 2. Intrinsics $K$ and extrinsics $[R \mid t]$

Real images are measured in **pixels** with the origin in the top-left corner:

$$ K = \begin{pmatrix} f_x & s & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{pmatrix} $$

$f_x, f_y$: focal length in pixels ($f_x = f/\text{pixel width}$), $(c_x, c_y)$: principal point, $s$: skew (≈ 0).
The pose of the camera transforms world coordinates into camera coordinates: $X_c = R\,X_w + t$. Altogether:

$$ \tilde{u} = K\,[R \mid t]\,\tilde{X}_w = P\,\tilde{X}_w, \qquad P \in \mathbb{R}^{3\times4}. $$

### Exercise 2.1
Complete `rotation` (Euler angles in degrees, $R = R_z R_y R_x$) and `project` (world points `3 x N` → pixels `2 x N`).

In [ ]:
def rotation(rx, ry, rz):
    rx, ry, rz = np.deg2rad([rx, ry, rz])
    # TODO
    ...


def project(Xw, K, R, t):
    """Xw: 3 x N world points -> 2 x N pixel coordinates (no distortion)."""
    # TODO
    ...


R_test = rotation(10, 20, 30)
assert np.allclose(R_test @ R_test.T, np.eye(3)) and np.isclose(np.linalg.det(R_test), 1)

### Exercise 2.2 — Explore
Use the sliders to move the camera. Observe: what does increasing $f$ do? What is the difference between moving the camera closer and increasing $f$ (look at the *perspective* of the cube)?

In [ ]:
from ipywidgets import FloatSlider, interact

cube = Xcube[:3]


@interact(f=FloatSlider(min=200, max=1500, value=500, step=50),
          tz=FloatSlider(min=3, max=20, value=6),
          tx=FloatSlider(min=-4, max=4, value=0),
          rx=FloatSlider(min=-60, max=60, value=20),
          ry=FloatSlider(min=-60, max=60, value=30))
def explore(f, tz, tx, rx, ry):
    K = np.array([[f, 0, 320], [0, f, 240], [0, 0, 1]])
    uv = project(cube, K, rotation(rx, ry, 0), [tx, 0, tz])
    plt.figure(figsize=(6, 4.5))
    plt.plot(uv[0], uv[1], "-o")
    plt.axis([0, 640, 480, 0])  # image of 640 x 480, y axis pointing down
    plt.gca().set_aspect("equal")
    plt.title(f"f={f:.0f}px, distance={tz:.1f}")
    plt.show()

### Exercise 2.3 — By hand (exam style)
$K = \begin{pmatrix} 800 & 0 & 320 \\ 0 & 800 & 240 \\ 0 & 0 & 1\end{pmatrix}$, $R = I$, $t = (0, 0, 500)$ mm. Compute **on paper** the pixel of the world point $X_w = (100, -50, 1000)$ mm, then check with `project` and with `cv2.projectPoints`.

In [ ]:
K_ex = np.array([[800.0, 0, 320], [0, 800, 240], [0, 0, 1]])
Xw = np.array([[100.0], [-50.0], [1000.0]])
t_ex = np.array([0.0, 0.0, 500.0])
# TODO
my_answer = (u, v)
uv = project(Xw, K_ex, np.eye(3), t_ex).ravel()
uv_cv, _ = cv2.projectPoints(Xw.T, np.zeros(3), t_ex, K_ex, None)
print("project:", uv, "| OpenCV:", uv_cv.ravel(), "| by hand:", np.array(my_answer))
assert np.allclose(uv, my_answer, atol=0.1) and np.allclose(uv, uv_cv.ravel())

## 3. Lens distortion

Real (especially wide-angle) lenses bend straight lines. OpenCV models the distortion on the **normalised** coordinates $x = X_c/Z_c$, $y = Y_c/Z_c$, with $r^2 = x^2 + y^2$:

$$ \begin{aligned} x_d &= x\,(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + 2p_1xy + p_2(r^2 + 2x^2) \\ y_d &= y\,(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + p_1(r^2 + 2y^2) + 2p_2xy \end{aligned} $$

then $u = f_x x_d + c_x$, $v = f_y y_d + c_y$. $k_i$: **radial** (barrel if $k_1 < 0$, pincushion if $k_1 > 0$), $p_i$: **tangential** (lens not parallel to the sensor).

### Exercise 3.1
Implement `distort` and visualise what a grid of straight lines becomes for $k_1 = -0.3$ and $k_1 = +0.3$. Check against `cv2.projectPoints`, which takes `dist = (k1, k2, p1, p2, k3)`.

In [ ]:
def distort(x, y, k1=0.0, k2=0.0, p1=0.0, p2=0.0, k3=0.0):
    # TODO
    ...


K_cam = np.array([[600.0, 0, 640], [0, 600, 360], [0, 0, 1]])
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, k1 in zip(ax, [0.0, -0.3, 0.3]):
    for c in np.linspace(-1, 1, 11):
        s = np.linspace(-1, 1, 100)
        for gx, gy in [(np.full_like(s, c), s * 0.6), (s, np.full_like(s, c * 0.6))]:
            xd, yd = distort(gx, gy, k1=k1)
            a.plot(K_cam[0, 0] * xd + K_cam[0, 2], K_cam[1, 1] * yd + K_cam[1, 2], "b", lw=0.8)
    a.axis([0, 1280, 720, 0]); a.set_aspect("equal"); a.set_title(f"k1 = {k1}")

pts = np.array([[0.4, -0.3, 1.0], [-0.7, 0.5, 1.0]])
dist = np.array([-0.3, 0.1, 0.001, -0.002, 0.0])
uv_cv, _ = cv2.projectPoints(pts, np.zeros(3), np.zeros(3), K_cam, dist)
xd, yd = distort(pts[:, 0], pts[:, 1], *dist)
assert np.allclose(uv_cv.reshape(-1, 2), np.c_[600 * xd + 640, 600 * yd + 360])

**Question.** Distortion is a function of $r$: where in the image are the errors largest? What does it imply for a camera looking at the corners of a work table?

## 4. Back-projection: from a pixel to the table

A pixel only defines a **ray** through the optical centre: depth is lost. If the point is known to lie on the table plane $Z_w = 0$, we can intersect the ray with that plane:

$$ \tilde{u} \sim K\,[r_1\ r_2\ r_3\ t]\,(X, Y, 0, 1)^T = \underbrace{K\,[r_1\ r_2\ t]}_{H}\,(X, Y, 1)^T $$

So between the table plane and the image there is a 3×3 invertible matrix $H$: a **homography**. $(X, Y, 1)^T \sim H^{-1}\tilde u$.

### Exercise 4.1
Implement `pixel_to_table(u, v, K, R, t)` with $H^{-1}$ and check the round trip on random table points.

In [ ]:
def pixel_to_table(uv, K, R, t):
    """uv: 2 x N pixels -> 2 x N points (X, Y) on the plane Z_w = 0 (no distortion)."""
    # TODO
    ...


R_cam, t_cam = rotation(170, 5, 0), np.array([-200.0, 150.0, 600.0])
XY = np.random.default_rng(0).uniform(0, 400, (2, 10))
uv = project(np.vstack([XY, np.zeros((1, 10))]), K_cam, R_cam, t_cam)
assert np.allclose(pixel_to_table(uv, K_cam, R_cam, t_cam), XY)
print("round trip OK")

## 5. Homography from 4 points

We usually do **not** know $K$, $R$, $t$. But a homography has only 8 degrees of freedom (9 entries up to scale: fix $h_{33} = 1$). A point $(X, Y)$ of the plane appears at the pixel $(u, v)$:

$$ u = \frac{h_{11}X + h_{12}Y + h_{13}}{h_{31}X + h_{32}Y + 1}, \qquad v = \frac{h_{21}X + h_{22}Y + h_{23}}{h_{31}X + h_{32}Y + 1} $$

Multiplying by the denominator and isolating $u$ and $v$ gives 2 equations that are **linear in the unknowns** $h_{ij}$ ($X, Y, u, v$ are known):

$$ \begin{aligned} u &= h_{11}X + h_{12}Y + h_{13} - h_{31}Xu - h_{32}Yu \\ v &= h_{21}X + h_{22}Y + h_{23} - h_{31}Xv - h_{32}Yv \end{aligned} $$

In matrix form, with $h = (h_{11}, h_{12}, h_{13}, h_{21}, h_{22}, h_{23}, h_{31}, h_{32})$, each point gives the two rows $(X, Y, 1, 0, 0, 0, -Xu, -Yu)\,h = u$ and $(0, 0, 0, X, Y, 1, -Xv, -Yv)\,h = v$.

→ **4 points** (no 3 aligned) give a linear system $A\,h = b$ of size 8×8. This is the *Direct Linear Transform* (DLT).

### Exercise 5.1 — Estimate H
Implement `homography_4pts(src, dst)` that returns the 3×3 matrix mapping `src` points onto `dst` points. Compare to `cv2.getPerspectiveTransform`.

In [ ]:
def homography_4pts(src, dst):
    """src, dst: arrays (4, 2). Returns H (3x3) with dst ~ H @ src."""
    A, b = [], []
    # TODO
    ...


src = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], float)
dst = np.array([[10, 20], [200, 30], [220, 180], [5, 160]], float)
H_mine = homography_4pts(src, dst)
assert np.allclose(H_mine, cv2.getPerspectiveTransform(src.astype(np.float32), dst.astype(np.float32)), atol=1e-6)
print(H_mine)

### Exercise 5.2 — Rectify the book cover
The 4 corners of the cover in `cover.jpg` are given below (top-left, top-right, bottom-right, bottom-left). The book measures about **155 × 230 mm**. Produce a flat image of the cover at 3 pixels/mm:

1. compute $H$ mapping the **flat** image coordinates to the photo,
2. for every pixel of the flat image, look up the colour at $H\,(x, y, 1)^T$ in the photo (*inverse mapping*: no holes!). Vectorise with `np.meshgrid`;
3. compare with `cv2.warpPerspective`.

Set `INTERACTIVE = True` to click the corners yourself (requires `ipympl`).

In [ ]:
cover = cv2.imread(str(DATA / "cover.jpg"))[:, :, ::-1]
corners = np.array([[357, 338], [690, 240], [1054, 572], [612, 816]], float)

INTERACTIVE = False
if INTERACTIVE:
    %matplotlib ipympl
    clicked = []
    fig = plt.figure()
    plt.imshow(cover)
    fig.canvas.mpl_connect("button_press_event",
                           lambda e: (clicked.append((e.xdata, e.ydata)), plt.plot(e.xdata, e.ydata, "rx")))
    # after clicking 4 corners: corners = np.array(clicked)

plt.imshow(cover); plt.plot(*np.vstack([corners, corners[:1]]).T, "r-o");

In [ ]:
ppm = 3
w, h = 155 * ppm, 230 * ppm
flat_corners = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], float)
# TODO
flat = ...  (h x w x 3 image)
flat_cv = cv2.warpPerspective(cover, cv2.getPerspectiveTransform(corners.astype(np.float32),
                                                                 flat_corners.astype(np.float32)), (w, h))
fig, ax = plt.subplots(1, 2, figsize=(8, 6))
ax[0].imshow(flat); ax[0].set_title("my rectification")
ax[1].imshow(flat_cv); ax[1].set_title("cv2.warpPerspective");

**Questions.**
1. Why do we map flat → photo and not photo → flat?
2. The homography is exact only for points **on the plane** of the cover. What happens to the thickness of the book? To a screw lying on the table in the project?

## 6. (Bonus) Fully automatic rectification

Detect the 4 corners of the A3 sheet (420 × 297 mm) in `document.jpg` automatically, then rectify it. Hint: the paper is bright and unsaturated, the table is brown → threshold in HSV, largest contour, `cv2.approxPolyDP`. Order the corners (top-left first, clockwise).

In [ ]:
doc = cv2.imread(str(DATA / "document.jpg"))


def order_corners(pts):
    """Order 4 points as top-left, top-right, bottom-right, bottom-left."""
    pts = pts.reshape(4, 2).astype(float)
    s, d = pts.sum(1), np.diff(pts, axis=1).ravel()
    return np.array([pts[np.argmin(s)], pts[np.argmin(d)], pts[np.argmax(s)], pts[np.argmax(d)]])


# TODO
doc_corners = ...  (4 x 2, ordered)
w, h = 420 * 2, 297 * 2
Hd = cv2.getPerspectiveTransform(doc_corners.astype(np.float32),
                                 np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]]))
flat_doc = cv2.warpPerspective(doc, Hd, (w, h))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(doc[:, :, ::-1]); ax[0].plot(*np.vstack([doc_corners, doc_corners[:1]]).T, "r-o")
ax[1].imshow(flat_doc[:, :, ::-1]);